In [3]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError("plotly is required. Install with: pip install plotly") from exc

CSV_PATH = "Data/2026-03-07-up__yearly.csv"

df = pd.read_csv(CSV_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = [
    "real_scrap",
    "real_scrap_savings",
    "predicted_scrap",
    "predicted_scrap_savings",
]
missing = [c for c in required_cols + ["date"] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

n = len(df)
if n == 0:
    raise ValueError("CSV is empty")

def pct_index(pct: float) -> int:
    return min(max(int(round((n - 1) * pct)), 0), n - 1)

prio_default = df.loc[pct_index(0.10), "date"].date()
impl_default = df.loc[pct_index(0.40), "date"].date()
curr_default = df.loc[pct_index(0.80), "date"].date()

min_date = df["date"].min().date()
max_date = df["date"].max().date()

prio_picker = widgets.DatePicker(description="DateOfPrio", value=prio_default)
impl_picker = widgets.DatePicker(description="ImplementedDate", value=impl_default)
curr_picker = widgets.DatePicker(description="CurrentDate", value=curr_default)

view_min_picker = widgets.DatePicker(description="MinViewDate", value=min_date)
view_max_picker = widgets.DatePicker(description="MaxViewDate", value=max_date)

out = widgets.Output()

def draw(_=None):
    with out:
        out.clear_output(wait=True)

        needed = [
            prio_picker.value,
            impl_picker.value,
            curr_picker.value,
            view_min_picker.value,
            view_max_picker.value,
        ]
        if not all(needed):
            print("Please select all dates.")
            return

        date_of_prio = pd.Timestamp(prio_picker.value)
        implemented_date = pd.Timestamp(impl_picker.value)
        current_date = pd.Timestamp(curr_picker.value)
        view_min = pd.Timestamp(view_min_picker.value)
        view_max = pd.Timestamp(view_max_picker.value)

        if not (date_of_prio <= implemented_date <= current_date):
            print("Invalid order: DateOfPrio <= ImplementedDate <= CurrentDate is required.")
            return

        if view_min > view_max:
            print("Invalid view range: MinViewDate must be <= MaxViewDate.")
            return

        d = df["date"]

        # Visibility rules
        # 1) Left of DateOfPrio and up to ImplementedDate: real_scrap
        real_scrap_visible = np.where(d < implemented_date, df["real_scrap"], np.nan)

        # 2) ImplementedDate..CurrentDate: real_scrap_savings + predicted lines
        implemented_to_current = (d >= implemented_date) & (d <= current_date)
        real_scrap_savings_visible = np.where(d >= implemented_date, df["real_scrap_savings"], np.nan)
        predicted_scrap_visible = np.where(implemented_to_current, df["predicted_scrap"], np.nan)
        predicted_scrap_savings_visible = np.where(implemented_to_current, df["predicted_scrap_savings"], np.nan)

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=d,
                y=real_scrap_visible,
                mode="lines",
                name="real_scrap",
                line={"width": 2},
                hovertemplate="%{x|%Y-%m-%d}<br>real_scrap=%{y}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=d,
                y=real_scrap_savings_visible,
                mode="lines",
                name="real_scrap_savings",
                line={"width": 2},
                hovertemplate="%{x|%Y-%m-%d}<br>real_scrap_savings=%{y}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=d,
                y=predicted_scrap_visible,
                mode="lines",
                name="predicted_scrap",
                line={"width": 2, "dash": "dash"},
                hovertemplate="%{x|%Y-%m-%d}<br>predicted_scrap=%{y}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=d,
                y=predicted_scrap_savings_visible,
                mode="lines",
                name="predicted_scrap_savings",
                line={"width": 2, "dash": "dot"},
                hovertemplate="%{x|%Y-%m-%d}<br>predicted_scrap_savings=%{y}<extra></extra>",
            )
        )

        for x, color, dash, label in [
            (date_of_prio, "gray", "dash", "DateOfPrio"),
            (implemented_date, "black", "dashdot", "ImplementedDate"),
            (current_date, "green", "dashdot", "CurrentDate"),
        ]:
            fig.add_vline(x=x, line_color=color, line_dash=dash, line_width=1.5)
            fig.add_annotation(
                x=x,
                y=1,
                yref="paper",
                yanchor="bottom",
                text=label,
                showarrow=False,
                font={"size": 11, "color": color},
            )

        fig.update_layout(
            title="Scrap Metrics Over Time",
            xaxis_title="Date",
            yaxis_title="Value",
            template="plotly_white",
            hovermode="x unified",
            # width=3400,
            height=650,
            # legend={"orientation": "h", "y": 1.08, "x": 0},
            # margin={"l": 60, "r": 30, "t": 80, "b": 60},
        )
        fig.update_xaxes(range=[view_min, view_max], rangeslider_visible=True)

        fig.show()

for w in (prio_picker, impl_picker, curr_picker, view_min_picker, view_max_picker):
    w.observe(draw, names="value")

display(widgets.VBox([
    widgets.HBox([prio_picker, impl_picker, curr_picker]),
    widgets.HBox([view_min_picker, view_max_picker]),
]))
display(out)
draw()
3


Output()

3